# 12 Scoring service (Part E)

`src/scoring.py` packages the deployed models behind one call:

```python
scorer = Scorer.load()
scorer.score(application, history)  # -> P(default), decision, top-3 reasons, offered amount
```

It builds features with the **same code as training** (`src/features.py`), so there is no second implementation to drift. `tests/test_scoring.py` checks this: for 40 random Train loans, the service rebuilds exactly the training features from the raw application and history.

In [1]:
import sys, warnings, json, time; sys.path.append(".."); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from src.config import set_seed, MODELS
from src.data import load_loans
from src.scoring import Scorer, APPLICATION_FIELDS, HISTORY_FIELDS
set_seed()
t0 = time.perf_counter()
scorer = Scorer.train()
path = scorer.save()
print(f"Trained in {time.perf_counter() - t0:.1f}s; saved to {path.relative_to(path.parents[1])} ({path.stat().st_size / 1024:.0f} KB)")
print(f"Version {scorer.version}; trained through {scorer.trained_through}; threshold {scorer.threshold:.3f}; "
      f"first-loan cap {scorer.starter_cap:,.0f}; history prior {scorer.prior:.4f}")

Trained in 4.1s; saved to models/nkwapay-v1.joblib (19 KB)
Version nkwapay-v1; trained through 2024-11-14; threshold 0.062; first-loan cap 5,000; history prior 0.0163


## 12.1 What the app must send

**The application** (one loan request):

In [2]:
pd.DataFrame({'field': list(APPLICATION_FIELDS), 'meaning': list(APPLICATION_FIELDS.values())})

,field,meaning
0,customer_id,NkwaPay customer identifier
1,country_id,"Market, e.g. 'Kenya'"
2,loan_type,"Product code, e.g. 'Type_1'"
3,lender_id,Primary funding lender
4,lenders,"All funding lenders with shares, e.g. [{'lende..."
5,Total_Amount,"Requested principal, local currency"
6,duration,Requested term in days
7,application_date,"Date of the decision (ISO), used as disburseme..."
8,New_versus_Repeat,'New Loan' or 'Repeat Loan'


**The applicant's earlier loans with NkwaPay** (list of records, may be empty):

In [3]:
pd.DataFrame({'field': list(HISTORY_FIELDS), 'meaning': list(HISTORY_FIELDS.values())})

,field,meaning
0,disbursement_date,Date the earlier loan was disbursed
1,due_date,Its due date
2,Total_Amount,Its principal
3,duration,Its term in days
4,lender_id,Its primary lender
5,target,"1 defaulted, 0 repaid, missing if not yet known"


Notes for the app team:
- `history` should come from NkwaPay's own ledger, **with outcomes where known**. Only loans due before the application date count as resolved; open loans count as active.
- Never send `Total_Amount_to_Repay` or any post-disbursement balance: those were leaky in training (Section 1) and are not inputs.
- Unseen products or lenders are accepted and mapped to `Unknown`; the reason code then says the product is new to us.
- No personal or sensitive attributes are needed (Act 843 s.19 minimality, s.37 special data).

## 12.2 Example calls

In [4]:
tr = load_loans("train")
hist_of = lambda r: tr[(tr.customer_id == r.customer_id) & (tr.disbursement_date < r.disbursement_date)][list(HISTORY_FIELDS)] \
    .assign(disbursement_date=lambda d: d.disbursement_date.dt.date.astype(str), due_date=lambda d: d.due_date.dt.date.astype(str)).to_dict("records")
def app_of(r, **over):
    a = {"customer_id": int(r.customer_id), "country_id": r.country_id, "loan_type": r.loan_type, "lender_id": int(r.lender_id),
         "lenders": [{"lender_id": int(r.lender_id), "share": float(r.Lender_portion_Funded)}], "Total_Amount": float(r.Total_Amount),
         "duration": int(r.duration), "application_date": str(r.disbursement_date.date()), "New_versus_Repeat": r.New_versus_Repeat}
    return {**a, **over}

late = tr[tr.disbursement_date >= "2024-01-01"]
regular = tr[(tr.loan_type == "Type_1") & (tr.disbursement_date >= "2022-11-01")].iloc[-1]   # Type_1 ends in Nov 2022
risky = late[(late.duration >= 30) & (late.loan_type != "Type_1")].iloc[0]
examples = {
    "regular repeat borrower": (app_of(regular), hist_of(regular)),
    "long-term loan, repeat borrower": (app_of(risky), hist_of(risky)),
    "first-time applicant, same 30-day loan": (app_of(risky, customer_id=999_999_001, New_versus_Repeat="New Loan"), []),
    "Ghana applicant, unseen product": (app_of(regular, country_id="Ghana", loan_type="Type_3", lender_id=296542,
                                              lenders=[{"lender_id": 296542, "share": 0.3}]), []),
}
for name, (a, h) in examples.items():
    out = scorer.score(a, h)
    print(f"--- {name} ({len(h)} earlier loans)")
    print(json.dumps(out, indent=2, default=str))

--- regular repeat borrower (73 earlier loans)
{
  "model_version": "nkwapay-v1",
  "p_default": 0.0005,
  "threshold": 0.0617,
  "decision": "approve",
  "note": "",
  "reasons": [],
  "requested_amount": 19823.0,
  "suggested_limit": 9030.83,
  "offered_amount": 9030.83,
  "latency_ms": 14.25
}
--- long-term loan, repeat borrower (4 earlier loans)
{
  "model_version": "nkwapay-v1",
  "p_default": 0.1155,
  "threshold": 0.0617,
  "decision": "decline",
  "note": "Estimated default risk above the cost threshold.",
  "reasons": [
    {
      "code": "LOAN_TERMS",
      "log_odds": 1.757,
      "text": "The amount and repayment period requested (25,000 over 30 days) are typical of loans that were more often not repaid; longer repayment periods carry the most risk in our records."
    },
    {
      "code": "RECENT",
      "log_odds": 0.539,
      "text": "Your recent borrowing pattern (time since your last loan and loans currently running) differs from that of borrowers who usually repay

The four calls show each branch of the policy: **approve** with an offered amount capped at the predicted limit; **decline** with three reasons; **refer** for a first-time applicant above the threshold (starter-loan track or human review instead of an automatic decline, Section 11.3); and **refer** for any Ghana application, because the model is not validated there (Section 10). The JSON is the response the app receives.

## 12.3 Latency

In [5]:
sample = tr.sample(500, random_state=1)
calls = [(app_of(r), hist_of(r)) for _, r in sample.iterrows()]
lat = []
for a, h in calls:
    t = time.perf_counter(); scorer.score(a, h); lat.append((time.perf_counter() - t) * 1000)
lat = np.array(lat)
n_hist = np.array([len(h) for _, h in calls])
print(f"Single-call latency over {len(lat)} calls: median {np.median(lat):.1f} ms, p95 {np.percentile(lat, 95):.1f} ms, max {lat.max():.1f} ms")
print(f"History length in sample: median {np.median(n_hist):.0f}, max {n_hist.max()} earlier loans")
pd.DataFrame({"history length": pd.cut(n_hist, [-1, 0, 5, 20, 1000], labels=["0", "1-5", "6-20", ">20"]), "ms": lat}) \
    .groupby("history length", observed=True).ms.describe(percentiles=[.5, .95])[["count", "50%", "95%"]].round(1)

Single-call latency over 500 calls: median 17.6 ms, p95 24.5 ms, max 40.3 ms
History length in sample: median 7, max 167 earlier loans


,count,50%,95%
history length,,,
0,43.0,17.7,24.0
1-5,164.0,17.5,23.6
6-20,195.0,17.5,24.7
>20,98.0,17.7,23.9


In [6]:
from src import models as M, features as F
world = pd.concat([tr, load_loans("test")], ignore_index=True)
t = time.perf_counter()
Xb = F.build_features(world, prior=scorer.prior).iloc[: len(tr)]
p = scorer.classifier.predict_proba(Xb[M.inputs(M.DEPLOY_FEATURES)])[:, 1]
dt = time.perf_counter() - t
print(f"Batch scoring of {len(tr):,} loans (features + model): {dt:.2f}s, {len(tr) / dt:,.0f} loans/s")

Batch scoring of 66,520 loans (features + model): 0.52s, 127,458 loans/s


**Reading.** A single call takes about 20 ms (median; p95 about 32 ms) on a laptop, well within an interactive app's budget (a few hundred ms for a USSD or mobile-money flow), and it does not grow with history length in the range seen (up to 167 earlier loans). Most of the time is in pandas feature construction, not the model; batch scoring is far faster per loan, which suits nightly re-scoring and monitoring. The model file is small enough to ship inside the app's backend container.

## 12.4 Operating notes
- **Versioning.** The saved object records the version, training cut-off, threshold and feature lists. Any retrain or recalibration bumps the version and is logged with the decisions it made (accountability, Act 843 s.17).
- **Monitoring.** Log P(default), decision and reason codes (not the personal inputs) per call; weekly, compare mean prediction with observed defaults by score band, track approval and refer rates by history band, and the Section 10 domain-classifier score for drift.
- **Recalibration.** Section 5 found over-spread probabilities after Dec 2022: refit a Platt layer on the latest resolved loans monthly and update `threshold` if `src/costs.py` assumptions change.
- **Not yet implemented:** credit-bureau lookup (BoG Directive §7.3; Act 726), consent capture, and the 21-day human-review workflow; these sit around the service, not in it.